
# Phase 5-2: 광주 전역 / 동구 Zero-Shot 추론 및 핫스팟 시각화
이 노트북은 앞서 훈련된 범용 모델(`universal_rf_model.pkl`)을 불러와서 타겟 격자망에 적용(Inference)하고,
추론된 확률값(0.0 ~ 1.0)을 지도상에 시각화하여 핫스팟 분포를 검증하는 프로토타입입니다.

> **Note:** 현재 동구 데이터 기준으로 시각화 결과가 다소 "어이없는" 상태(과적합 또는 구분 실패)일 수 있습니다.
> 추후 모델 피처(Feature) 보강이나 앙상블 파라미터 조정을 통해 모델을 고도화해야 합니다.


In [ ]:

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 경로 설정
base_dir = Path('../../')
model_path = base_dir / 'data/models/universal_rf_model.pkl'

# 현재는 광주 전체 격자가 없으므로, 생성해둔 동구 격자로 먼저 시각화 검증을 진행합니다.
data_path = base_dir / 'data/processed/dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg'

# 커스텀 클래스(PUBaggingRandomForest) 역직렬화를 위한 클래스 선언
# (이게 없으면 joblib.load 시 AttributeError가 발생합니다)
from sklearn.ensemble import RandomForestClassifier

class PUBaggingRandomForest:
    def __init__(self, n_estimators=10, random_state=42):
        self.n_estimators = n_estimators
        self.models = []
        self.random_state = random_state
        
    def fit(self, X, y):
        pass # 추론용이므로 Pass
            
    def predict_proba(self, X):
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        return probas.mean(axis=0)

print("✅ 라이브러리 로드 및 클래스 선언 완료")



## 1. 모델 및 타겟 데이터 로드


In [ ]:

print(f"📦 모델 로드 중... ({model_path.name})")
model = joblib.load(model_path)

print(f"🗺️ 타겟 공간 데이터 로드 중... ({data_path.name})")
gdf = gpd.read_file(data_path)

# 모델 추론을 위한 X 피처 추출
feature_cols = [col for col in gdf.columns if '_count_' in col]
X = gdf[feature_cols].values

print(f"추출된 피처 ({len(feature_cols)}개): {feature_cols}")
print(f"추론할 타겟 격자 수: {len(X):,}개")



## 2. Zero-Shot 추론 수행 (Inference)


In [ ]:

print("🧠 추론(Inference) 진행 중...")
trash_score = model.predict_proba(X)

# 데이터프레임에 확률값 삽입
gdf['trash_score'] = trash_score

print(f"예측 확률 점수 분포: 최소 {trash_score.min():.4f} ~ 최대 {trash_score.max():.4f}")
print("상위 10개 점수:", np.sort(trash_score)[-10:])



## 3. 핫스팟 분포 지도 시각화


In [ ]:

print("🎨 시각화 생성 중...")

fig, ax = plt.subplots(1, 1, figsize=(15, 15))

# 핫스팟 히트맵 렌더링
gdf.plot(
    column='trash_score',
    ax=ax,
    cmap='YlOrRd',  # 노란색 -> 주황색 -> 빨간색
    legend=True,
    legend_kwds={'label': "Trash Spot Probability", 'orientation': "horizontal", 'shrink': 0.6},
    alpha=0.9,
    edgecolor='none'
)

# 배경색을 어둡게 처리하여 핫스팟 대비 효과 극대화
ax.set_facecolor('#1e1e1e')
fig.patch.set_facecolor('#1e1e1e')
ax.set_title("Hotspot Prediction Map (PU-Learning)", color='white', fontsize=18, pad=20)
ax.axis('off')

# 레전드 폰트 컬러 조정
if ax.get_legend():
    plt.setp(ax.get_legend().get_texts(), color='white')

plt.show()
